# Exploration Notebook: Conversational Book Recommendation Assistant using RAG

This notebook documents the exploratory analysis and testing behind the project pipeline.

It focuses on:
- understanding how conversational user requests are parsed into retrieval-friendly tags
- testing book retrieval from Google Books and Open Library
- inspecting normalization and deduplication
- checking text quality filtering
- evaluating the ranking logic used for final recommendations
- running the end-to-end recommendation pipeline on sample queries

## 1. Goal of the exploration

Public book APIs often perform poorly when users type long conversational prompts.

Example:
- *"I want a motivating self-help book that helps me become more disciplined and improve my habits"*

A direct API search with that entire sentence may return noisy or weak results.

The goal of this notebook is to explore a better workflow:
1. parse the natural-language request into structured tags
2. use those tags to create short retrieval queries
3. fetch candidate books from external APIs
4. normalize and deduplicate the results
5. filter out weak or unusable items
6. rank the remaining books based on semantic similarity and tag matches


In [1]:
# Standard library
import os
import sys
import json
import pprint
from pathlib import Path

# Make project imports work when running from /notebooks
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print("Project root:", project_root)

Project root: /Users/sanjeevani1109/Desktop/Projects/Conversational Book Recommendation Assistant using RAG


In [2]:
# Project imports
from src.query_parser import rewrite_query_with_llm, fallback_tag_parse
from src.pipeline import (
    build_search_tag_groups,
    build_api_queries,
    collect_google_results,
    collect_openlibrary_results,
    enrich_open_library_books_with_descriptions,
    build_rerank_query,
    get_recommendations,
)
from src.normalizer import normalize_google_books, normalize_open_library
from src.deduplicator import deduplicate_books
from src.filtering import filter_books_with_usable_text, build_book_text
from src.embedder import load_embedding_model, embed_books, embed_query
from src.ranker import rank_books_by_similarity
from src.explainer import explain_recommendations


## 2. Define a sample user query

Start with a natural-language request similar to how a real user would interact with the app.

In [3]:
user_query = "I want a motivating self-help book that helps me become more disciplined and improve my habits"
print(user_query)

I want a motivating self-help book that helps me become more disciplined and improve my habits


## 3. Parse the query into structured tags

The project first uses an LLM-based parser to convert the user's request into structured fields:
- `primary_search_tags`
- `secondary_search_tags`
- `vibe_tags`
- `must_have_tropes`
- `avoid_terms`

If the LLM output fails, the fallback parser can still provide sensible tags for common query types.


In [4]:
# LLM-based parsing
parsed_query = rewrite_query_with_llm(user_query)
parsed_query

{'primary_search_tags': ['self-help', 'discipline', 'habit building'],
 'secondary_search_tags': ['productivity',
  'personal development',
  'motivation'],
 'vibe_tags': ['motivational', 'practical', 'actionable'],
 'must_have_tropes': [],
 'avoid_terms': ['fiction', 'romance']}

In [5]:
# Optional: compare with the fallback rule-based parser
fallback_parsed = fallback_tag_parse(user_query)
fallback_parsed

{'primary_search_tags': ['self-discipline', 'self-control', 'habit building'],
 'secondary_search_tags': ['productivity',
  'behavior change',
  'personal development',
  'routines'],
 'vibe_tags': ['practical', 'motivational', 'actionable'],
 'must_have_tropes': [],
 'avoid_terms': ['fiction', 'romance']}

### Observation

At this stage, the system turns a long conversational request into compact retrieval-friendly tags.
This is important because external APIs usually respond better to short topic-based keywords than to full natural-language paragraphs.


## 4. Build grouped search tags and API queries

The parsed tags are cleaned and grouped into categories. Then the system creates a small set of short API queries for retrieval.


In [6]:
tag_groups = build_search_tag_groups(parsed_query, user_query)
tag_groups

{'primary': ['self-help', 'discipline', 'habit building'],
 'secondary': ['productivity', 'personal development', 'motivation'],
 'vibe': ['motivational', 'practical', 'actionable'],
 'must_have_tropes': [],
 'avoid_terms': ['fiction', 'romance'],
 'fallback': ['I want a motivating self-help book that helps me become more disciplined and improve my habits']}

In [7]:
search_queries = build_api_queries(tag_groups)
search_queries

['self-help',
 'discipline',
 'habit building',
 'self-help discipline',
 'self-help habit building',
 'self-help productivity',
 'productivity']

### Observation

These short search queries are more suitable for Google Books and Open Library than the original full sentence.
The retrieval step intentionally uses several small queries instead of one long prompt.


## 5. Retrieve candidate books from external APIs

The project currently uses:
- Google Books
- Open Library

The cells below fetch a small number of candidate results for exploratory inspection.

In [8]:
google_raw = collect_google_results(search_queries, max_total=10, per_query=2)
openlibrary_raw = collect_openlibrary_results(search_queries, max_total=8, per_query=2)

print("Google raw results:", len(google_raw))
print("Open Library raw results:", len(openlibrary_raw))

Google raw results: 10
Open Library raw results: 8


In [9]:
# Inspect one raw item from each source
if google_raw:
    print("Sample Google Books item:")
    pprint.pp(google_raw[0])

if openlibrary_raw:
    print("Sample Open Library item:")
    pprint.pp(openlibrary_raw[0])

Sample Google Books item:
{'kind': 'books#volume',
 'id': 'qUAzAQAAMAAJ',
 'etag': '2xhJ9WgnxsE',
 'selfLink': 'https://www.googleapis.com/books/v1/volumes/qUAzAQAAMAAJ',
 'volumeInfo': {'title': 'Self-help',
                'subtitle': 'With Illustrations of Character, Conduct, and '
                            'Perseverance',
                'authors': ['Samuel Smiles'],
                'publishedDate': '1881',
                'description': 'Carl J. Martinson collection.',
                'industryIdentifiers': [{'type': 'OTHER',
                                         'identifier': 'UCAL:B4897020'}],
                'readingModes': {'text': False, 'image': True},
                'pageCount': 462,
                'printType': 'BOOK',
                'categories': ['Conduct of life'],
                'maturityRating': 'NOT_MATURE',
                'allowAnonLogging': False,
                'contentVersion': '0.6.7.0.full.1',
                'panelizationSummary': {'containsEpubBubbl

## 6. Normalize the results

Because Google Books and Open Library return different schemas, the project maps both into a shared structure with fields such as:
- title
- author
- description
- categories
- isbn
- published_year
- language
- cover_url
- source
- info_link

In [10]:
google_books = normalize_google_books(google_raw)
openlibrary_books = normalize_open_library(openlibrary_raw, detail_lookup={})

print("Normalized Google Books:", len(google_books))
print("Normalized Open Library:", len(openlibrary_books))

Normalized Google Books: 10
Normalized Open Library: 8


In [13]:
# Inspect normalized examples
if google_books:
    print(google_books[0])

{'title': 'Self-help', 'author': 'Samuel Smiles', 'description': 'Carl J. Martinson collection.', 'categories': 'Conduct of life', 'isbn': '', 'published_year': '1881', 'language': 'en', 'cover_url': 'http://books.google.com/books/content?id=qUAzAQAAMAAJ&printsec=frontcover&img=1&zoom=1&edge=curl&source=gbs_api', 'source': 'google_books', 'source_id': 'qUAzAQAAMAAJ', 'work_key': '', 'info_link': 'https://play.google.com/store/books/details?id=qUAzAQAAMAAJ&source=gbs_api'}


In [14]:
if openlibrary_books:
    print(openlibrary_books[0])

{'title': 'Self-help', 'author': 'Samuel Smiles', 'description': '', 'categories': '', 'isbn': '', 'published_year': '1800', 'language': 'pan', 'cover_url': 'https://covers.openlibrary.org/b/id/5587999-M.jpg', 'source': 'open_library', 'source_id': '/works/OL2321834W', 'work_key': '/works/OL2321834W', 'info_link': 'https://openlibrary.org/works/OL2321834W'}


### Observation

Normalization makes the downstream pipeline much easier because later steps can work with one common book format rather than source-specific fields.

## 7. Merge and deduplicate books

The same book can appear multiple times across APIs. The project deduplicates first by ISBN when available, and otherwise by normalized title-author pairs.

In [15]:
all_books = google_books + openlibrary_books
unique_books = deduplicate_books(all_books)

print("Books before deduplication:", len(all_books))
print("Books after deduplication:", len(unique_books))

Books before deduplication: 18
Books after deduplication: 16


In [16]:
# Show the first few deduplicated books
unique_books[:3]

[{'title': 'Self-help',
  'author': 'Samuel Smiles',
  'description': 'Carl J. Martinson collection.',
  'categories': 'Conduct of life',
  'isbn': '',
  'published_year': '1881',
  'language': 'en',
  'cover_url': 'http://books.google.com/books/content?id=qUAzAQAAMAAJ&printsec=frontcover&img=1&zoom=1&edge=curl&source=gbs_api',
  'source': 'google_books',
  'source_id': 'qUAzAQAAMAAJ',
  'work_key': '',
  'info_link': 'https://play.google.com/store/books/details?id=qUAzAQAAMAAJ&source=gbs_api'},
 {'title': 'Self-Help to ICSE Semester 2 Topicwise Revision Biology Book Class 10 (Subjective & Objective Format)',
  'author': 'Sunil Manchanda',
  'description': "Just as a guide leads an inquisitive traveller to his goal and while escorting him, narrated the salient features of the object, so does a good guide-book offers the students all the essential information for easy comprehension of the subject to prepare for the Final-Based Examination of Semester-II. 'Self-Help to I.C.S.E. Semester 

## 8. Enrich Open Library descriptions

Some Open Library search results do not include useful descriptions at the initial search stage.
The pipeline optionally performs a second lookup for a limited number of works to improve the text available for ranking.

In [17]:
enriched_books = enrich_open_library_books_with_descriptions(unique_books, max_enrich=4)
print("Books after enrichment:", len(enriched_books))

Books after enrichment: 16


In [18]:
# Inspect books after enrichment
enriched_books[:3]

[{'title': 'Self-help',
  'author': 'Samuel Smiles',
  'description': 'Carl J. Martinson collection.',
  'categories': 'Conduct of life',
  'isbn': '',
  'published_year': '1881',
  'language': 'en',
  'cover_url': 'http://books.google.com/books/content?id=qUAzAQAAMAAJ&printsec=frontcover&img=1&zoom=1&edge=curl&source=gbs_api',
  'source': 'google_books',
  'source_id': 'qUAzAQAAMAAJ',
  'work_key': '',
  'info_link': 'https://play.google.com/store/books/details?id=qUAzAQAAMAAJ&source=gbs_api'},
 {'title': 'Self-Help to ICSE Semester 2 Topicwise Revision Biology Book Class 10 (Subjective & Objective Format)',
  'author': 'Sunil Manchanda',
  'description': "Just as a guide leads an inquisitive traveller to his goal and while escorting him, narrated the salient features of the object, so does a good guide-book offers the students all the essential information for easy comprehension of the subject to prepare for the Final-Based Examination of Semester-II. 'Self-Help to I.C.S.E. Semester 

## 9. Filter books with usable text

Some retrieved items are weak for ranking because they contain very little text or look like non-book material such as journals, bulletins, or proceedings.

The filtering step keeps only books with enough usable text for semantic comparison.

In [19]:
usable_books = filter_books_with_usable_text(enriched_books)

print("Usable books after filtering:", len(usable_books))

Usable books after filtering: 15


In [20]:
# Inspect the text used for matching and ranking
if usable_books:
    sample_book = usable_books[0]
    print(build_book_text(sample_book)[:1000])

Self-help Samuel Smiles Conduct of life Carl J. Martinson collection.


### Observation

This step improves recommendation quality because semantic ranking becomes much less reliable when titles and descriptions are missing or when the item is not really a relevant book.

## 10. Load the embedding model

The project uses a sentence-transformer embedding model to represent both the rerank query and the candidate books as vectors.

Model used in the codebase:
- `all-MiniLM-L6-v2`

In [21]:
model = load_embedding_model()
print("Embedding model loaded successfully")

Embedding model loaded successfully


## 11. Build the rerank query

The rerank query combines:
- the original user request
- must-have tropes, if any
- vibe tags, if any

This helps ranking reflect both the main topic and the desired reading experience.

In [23]:
rerank_query = build_rerank_query(user_query, tag_groups)
rerank_query

'I want a motivating self-help book that helps me become more disciplined and improve my habits | Desired vibe: motivational, practical, actionable'

## 12. Embed books and query, then rank results

The ranker combines:
- semantic similarity between the query embedding and book embeddings
- keyword bonuses for primary, secondary, trope, and vibe matches
- penalties for avoid-term matches

In [24]:
book_embeddings = embed_books(model, usable_books)
query_embedding = embed_query(model, rerank_query)

top_books = rank_books_by_similarity(
    query_embedding=query_embedding,
    book_embeddings=book_embeddings,
    books=usable_books,
    top_k=5,
    primary_tags=tag_groups["primary"],
    secondary_tags=tag_groups["secondary"],
    must_have_tropes=tag_groups["must_have_tropes"],
    vibe_tags=tag_groups["vibe"],
    avoid_terms=tag_groups["avoid_terms"],
)

len(top_books)

5

In [26]:
for i, book in enumerate(top_books, start=1):
    print(f"{i}. {book['title']} by {book['author']}")
    print(f"Source: {book['source']}")
    print(f"Semantic score: {book['similarity_score']:.4f}")
    print(f"Final score: {book['final_score']:.4f}")
    print(f"Primary hits: {book['primary_hits']}")
    print(f"Secondary hits: {book['secondary_hits']}")
    print(f"Must-have hits: {book['must_have_hits']}")
    print(f"Vibe hits: {book['vibe_hits']}")
    print(f"Avoid hits: {book['avoid_hits']}")
    print(f"Categories: {book['categories']}")
    print(f"Link: {book['info_link']}")

1. Effortless Self-Help & Mindfulness by SR Gama
Source: google_books
Semantic score: 0.6004
Final score: 0.7254
Primary hits: 1
Secondary hits: 2
Must-have hits: 0
Vibe hits: 1
Avoid hits: 0
Categories: Self-Help
Link: https://play.google.com/store/books/details?id=3IBSEQAAQBAJ&source=gbs_api
2. The Habit Building Blueprint by 
Source: google_books
Semantic score: 0.5381
Final score: 0.6981
Primary hits: 2
Secondary hits: 1
Must-have hits: 0
Vibe hits: 2
Avoid hits: 0
Categories: Business & Economics
Link: https://play.google.com/store/books/details?id=rsCtEQAAQBAJ&source=gbs_api
3. 365 Prompts for Self-Discovery: Self Help Book for Personal Transformation by Egomerit LLC
Source: google_books
Semantic score: 0.5094
Final score: 0.5594
Primary hits: 1
Secondary hits: 0
Must-have hits: 0
Vibe hits: 0
Avoid hits: 0
Categories: Self-Help
Link: https://play.google.com/store/books/details?id=fMk7EQAAQBAJ&source=gbs_api
4. Hooked: How to Build Habit-Forming Products by Atina Amrahs
Source: g

### Observation

The final score is not based on embeddings alone.
The project adds lightweight heuristic boosts and penalties so that books matching explicit tags are rewarded, while books containing avoid terms are penalized.

## 13. Generate a natural-language explanation

After ranking the books, the project uses the LLM again to explain why the final recommendations match the user's request.

In [27]:
explanation = explain_recommendations(user_query, top_books)
print(explanation)

Okay, based on your request for a motivating self-help book to help you become more disciplined and improve your habits, here are the top recommendations with explanations of why they’re a good fit:

**1. The Habit Building Blueprint by [Author Name/Publisher]:** This book seems like the *most* directly relevant choice. It’s explicitly focused on habit formation – a core element of discipline. It breaks down the science behind habits (the Habit Loop) and gives you actionable strategies like the 2-Minute Rule and “If-Then” planning to build good habits and break bad ones. It’s a very structured, science-backed approach, which can be really effective for someone wanting to build a system.

**2. 365 Prompts for Self-Discovery by Egomerit LLC:** This book is a fantastic option if you’re looking for a more reflective and personal approach to building discipline. It’s a daily journaling prompt guide designed to help you understand yourself better, which often leads to more sustainable change

## 14. End-to-end pipeline test

This cell runs the entire pipeline in one step, similar to how the main app works.

In [28]:
result = get_recommendations(
    user_query=user_query,
    model=model,
    explain_fn=explain_recommendations,
)

result.keys()

Open Library query failed for 'self-help': 503 Server Error: Service Unavailable for url: https://openlibrary.org/search.json?q=self-help&limit=2
Open Library query failed for 'discipline': 503 Server Error: Service Unavailable for url: https://openlibrary.org/search.json?q=discipline&limit=2
Open Library query failed for 'habit building': 503 Server Error: Service Unavailable for url: https://openlibrary.org/search.json?q=habit+building&limit=2
Open Library query failed for 'self-help discipline': 503 Server Error: Service Unavailable for url: https://openlibrary.org/search.json?q=self-help+discipline&limit=2


dict_keys(['parsed_query', 'tag_groups', 'search_queries', 'top_books', 'explanation', 'stats'])

In [29]:
print("Parsed query:")
pprint.pp(result["parsed_query"])

print("Search queries:")
for q in result["search_queries"]:
    print("-", q)

print("Stats:")
pprint.pp(result["stats"])

Parsed query:
{'primary_search_tags': ['self-help', 'discipline', 'habit building'],
 'secondary_search_tags': ['productivity',
                           'personal development',
                           'motivation'],
 'vibe_tags': ['motivational', 'practical', 'actionable'],
 'must_have_tropes': [],
 'avoid_terms': ['fiction']}
Search queries:
- self-help
- discipline
- habit building
- self-help discipline
- self-help habit building
- self-help productivity
- productivity
Stats:
{'google_raw': 10,
 'openlibrary_raw': 0,
 'all_books': 10,
 'unique_books': 10,
 'usable_books': 10}


In [30]:
print("Top recommendations:")
for i, book in enumerate(result["top_books"], start=1):
    print(f"{i}. {book['title']} by {book['author']}")

Top recommendations:
1. Effortless Self-Help & Mindfulness by SR Gama
2. The Habit Building Blueprint by 
3. 365 Prompts for Self-Discovery: Self Help Book for Personal Transformation by Egomerit LLC


## 15. Test additional query examples

To see how robust the system is, it is useful to try multiple user intents.
Examples below include both nonfiction and fiction-style requests.

In [31]:
test_queries = [
    "books that help with discipline and consistency",
    "emotional books about grief that feel healing",
    "dark academia mystery with beautiful writing",
    "fantasy with political intrigue and morally grey characters",
]
test_queries

['books that help with discipline and consistency',
 'emotional books about grief that feel healing',
 'dark academia mystery with beautiful writing',
 'fantasy with political intrigue and morally grey characters']

In [32]:
for q in test_queries:
    print("" + "="*100)
    print("USER QUERY:", q)
    parsed = rewrite_query_with_llm(q)
    tag_groups = build_search_tag_groups(parsed, q)
    search_queries = build_api_queries(tag_groups)
    print("PARSED:")
    pprint.pp(parsed)
    print("SEARCH QUERIES:", search_queries)

USER QUERY: books that help with discipline and consistency
PARSED:
{'primary_search_tags': ['self-discipline', 'habit building', 'consistency'],
 'secondary_search_tags': ['productivity',
                           'time management',
                           'personal development'],
 'vibe_tags': ['practical', 'motivational', 'actionable'],
 'must_have_tropes': [],
 'avoid_terms': ['fiction', 'romance']}
SEARCH QUERIES: ['self-discipline', 'habit building', 'consistency', 'self-discipline habit building', 'self-discipline consistency', 'self-discipline productivity', 'productivity']
USER QUERY: emotional books about grief that feel healing
PARSED:
{'primary_search_tags': ['grief', 'healing', 'emotional'],
 'secondary_search_tags': ['loss', 'coping', 'mental health'],
 'vibe_tags': ['emotional', 'reflective', 'comforting'],
 'must_have_tropes': [],
 'avoid_terms': ['thriller', 'dark']}
SEARCH QUERIES: ['grief', 'healing', 'emotional', 'grief healing', 'grief emotional', 'grief loss',

## 16. Findings from exploration

Based on the experiments in this notebook, several key findings emerged:

1. **Short retrieval tags work better than raw conversational prompts.**  
   Public book APIs generally return more relevant books when the search is based on compact topic tags rather than full sentences.

2. **Multi-source retrieval improves coverage.**  
   Google Books and Open Library often complement each other, so combining them improves the candidate pool.

3. **Normalization and deduplication are essential.**  
   Without them, the pipeline would return inconsistent records and repeated books.

4. **Description quality matters for ranking.**  
   Enriching missing descriptions and removing books with weak text improves semantic ranking quality.

5. **A hybrid ranking strategy works well for a lightweight prototype.**  
   Combining embedding similarity with small tag-based bonuses produces more grounded recommendations than embeddings alone.


## 17. Conclusion

This exploration notebook shows how the project moved from a simple idea—*"recommend books from a user prompt"*—to a more structured and reliable RAG-style pipeline.

Instead of relying only on generation, the system:
- interprets the user request
- retrieves candidate books from external sources
- cleans and enriches the data
- reranks results using semantic similarity and tag matching
- explains the final recommendations in natural language

This makes the system a stronger portfolio project because it demonstrates not just prompting, but full pipeline thinking across NLP, retrieval, preprocessing, ranking, and application design.